# Rotation Fit Probability — ANN vs Random Forest (v2)

Predicts the probability that an employee will succeed **if rotated into a specific role**.

All logic lives in the `rotation_fit` package, so this notebook, the command line
and the webapp share exactly the same code. The previous all-in-one notebook is kept
in `notebooks/legacy/` for reference.

**Data sources** (paths in `config.yaml`)
- `Career_Transaction_Karyawan_2012-2026_excel.xlsx` — movement history
- `Dataset.xlsx` — employee attributes and performance ratings
- optional: Position → Job Family mapping, HRIS gender / marital status

**Target**: `success_fair` — did the employee outperform the typical 2025 outcome of
people who started 2024 at the same rating level. This removes the company-wide rating
recalibration that otherwise dominates the label.

> Set `USE_SYNTHETIC = True` below to try the notebook on randomly generated data.

# Setup & Config

In [ ]:
import os
import sys
from pathlib import Path

# run from the repo root, whether the notebook was opened from notebooks/ or the root
ROOT = Path.cwd() if (Path.cwd() / "rotation_fit").exists() else Path.cwd().parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import Image, display

from rotation_fit import RotationScorer, load_config, load_hr_data, train
from rotation_fit.cohort import build_cohort
from rotation_fit.report import save_outputs, write_report

USE_SYNTHETIC = False   # True = fake data with the real layout

overrides = {}
if USE_SYNTHETIC:
    from rotation_fit.synthetic import make_synthetic, synthetic_config_overrides
    overrides = synthetic_config_overrides(make_synthetic("data/synthetic"))
    overrides["paths"].update({"outputs": "outputs/synthetic", "report": "reports/synthetic"})

cfg = load_config(overrides=overrides)
pd.set_option("display.width", 160)
cfg["experiment"]

# Data Load & Cohort

In [ ]:
hr = load_hr_data(cfg)
for note in hr.notes:
    print("NOTE:", note)

cohort, funnel, by_rating = build_cohort(hr, cfg)
print("INCLUSION FUNNEL")
for k, v in funnel.items():
    print(f"  {k:<46} {v:>6,}")
by_rating

# Model Development

Hold-out set first (touched once, at the end); the split-ratio × seed × scaler grid runs
on the remaining development rows. Takes a few minutes with the full grid.

In [ ]:
result = train(cfg, hr=hr, progress=print)
result.summary.round(3)

# Model Evaluation (hold-out)

In [ ]:
print(f"best on development set: {result.best['model']} / {result.best['scaler']}")
print(f"baseline PR-AUC (guessing) = {result.y.mean():.3f}")
result.holdout.round(3)

# Insight Extraction

* **Significance** — drop-one likelihood-ratio test per attribute, Holm-corrected.
  `direction` + means higher values go with more success. Association, not cause.
* **Importance** — how much the chosen model leans on each attribute, across splits.
* **Significant-attributes model** — selection redone inside every split, then compared
  with the full model on the same splits.

In [ ]:
result.significance.round(4)

In [ ]:
result.importance.round(4)

In [ ]:
r = result.reduced
print("significant attributes:", r["features"])
print(f"loss vs full model: {r['mean_loss']:+.3f} (SE {r['se_loss']:.3f}); "
      f"tolerance {r['tolerance']} -> deployed: {result.artifact['metadata']['deployed_variant']}")
result.subgroups

# Save Model & Report

* `outputs/` — model file and engineered cohort: **personal data, keep local**.
* `reports/latest/` — aggregates and charts only: **safe to share**.

In [ ]:
paths = save_outputs(result, cfg["paths"]["outputs"])
report = write_report(result, cfg["paths"]["report"])
print(paths, report, sep="\n")
display(Image(str(report / "evaluation.png")))
display(Image(str(report / "importance.png")))

# Score a New Rotation

In [ ]:
scorer = RotationScorer(result.artifact, hr)
unit = scorer.destination_units()[0]
position = scorer.destination_positions()[0]
print("destination:", position, "@", unit)

ranking = scorer.rank_candidates(position, unit)
ranking[["NIK", "origin_position", "success_pct", "out_of_distribution"]].head(10)

In [ ]:
# why: percentage points gained / lost vs a typical value, for the top candidate
scorer.explain(ranking.head(1)[["NIK", "destination_position", "destination_org_unit"]])